# Sentinel-1 flood mapping of the 2024 Kenya long-rains floods

Regenerates every number and figure in the paper from public data. Each stage writes to `data/`, `outputs/` or `paper/` and caches large intermediates in `data/cache/` (gitignored), so a second run skips the downloads. Flood masks and the day-gap table are reused when present; delete `outputs/masks/` and `outputs/day_gap_sensitivity.csv` to recompute them. The U-Net stage runs on a Kaggle GPU (see `kaggle/`); this notebook uses the weights it produces in `models/`, or trains locally if `TRAIN_LOCALLY` is set.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.insert(0, str(ROOT / 'pipeline'))

import pandas as pd
pd.set_option('display.width', 160)
TRAIN_LOCALLY = False

## 1. UNOSAT reference

Catalogue every FL20240426KEN product on HDX, download the cumulative shapefile package, and rasterise flood water to the 10 m UTM 37S grid.

In [2]:
import fetch_reference, reference, tana_corridor

fetch_reference.main()
if not reference.TANA_TILES.exists():
    tana_corridor.main()
reference.main()

                                                                                     hdx_name      sensor reference_date  reported_flooded_km2  reported_structures  shp_bytes
3        cumulative-satellite-detected-waters-and-impact-in-kenya-between-24-to-28-april-2024  NOAA-VIIRS     2024-04-24                3800.0                  NaN   20741579
10                              flood-impact-analysis-in-kiambu-county-kenya-as-of-1-may-2024    Pléiades     2024-05-01                  13.0               1815.0   60544290
4                   flood-impact-analysis-in-nairobi-and-kiambu-county-kenya-as-of-1-may-2024    Pléiades     2024-05-01                  17.0               3663.0   59835613
5                 flood-impact-analysis-in-nairobi-and-machakos-county-kenya-as-of-1-may-2024    Pléiades     2024-05-01                  22.0               4246.0   59835613
1                                     flood-impact-analysis-in-nairobi-kenya-as-of-1-may-2024    Pléiades     2024-05-01     

          aoi    group     sensor reference_date  domain_km2  flood_km2  excluded_cloud_km2
   garissa_s2 riverine Sentinel-2     2024-04-29      161.57      27.94                0.54
   garissa_pl riverine   Pléiades     2024-05-05      107.22      21.74               -0.02
   nairobi_pl    urban   Pléiades     2024-05-01      506.44      59.69               73.25
tana_420_9600 riverine Sentinel-2     2024-04-29       99.86       0.00               -0.00
tana_440_9660 riverine Sentinel-2     2024-04-29      195.95       3.54                0.29
tana_460_9640 riverine Sentinel-2     2024-04-29       46.74       0.00                0.01
tana_460_9660 riverine Sentinel-2     2024-04-29      187.51       5.87               -0.01
tana_480_9640 riverine Sentinel-2     2024-04-29       47.30       0.00                0.01
tana_480_9660 riverine Sentinel-2     2024-04-29      176.79       4.89               -0.01
tana_500_9660 riverine Sentinel-2     2024-04-29      114.10       2.59         

## 2. Sentinel-1 passes

Post-event pass nearest each reference date and a same-orbit dry-season pass.

In [3]:
import pick_scenes

pick_scenes.main()
pd.read_csv(pick_scenes.SCENES).query("~aoi.str.startswith('tana_')")

          aoi reference_date  post_date  day_gap  headline   pre_date  post_cover  pre_cover  relative_orbit orbit_state  n_passes_within_gap
   garissa_s2     2024-04-29 2024-04-26       -3      True 2024-02-26       1.000      1.000             159   ascending                    1
   garissa_pl     2024-05-05 2024-05-08        3      True 2024-02-26       1.000      1.000             159   ascending                    1
   nairobi_pl     2024-05-01 2024-05-01        0      True 2024-02-19       1.000      1.000              57   ascending                    2
tana_420_9600     2024-04-29 2024-05-01        2      True 2024-02-19       1.000      1.000              57   ascending                    1
tana_440_9660     2024-04-29 2024-05-01        2      True 2024-02-19       1.000      1.000              57   ascending                    1
tana_460_9640     2024-04-29 2024-05-01        2      True 2024-02-19       1.000      1.000              57   ascending                    1
tana_4

,aoi,reference_date,post_id,post_date,day_gap,headline,pre_id,pre_date,post_cover,pre_cover,relative_orbit,orbit_state,n_passes_within_gap
0,garissa_s2,2024-04-29,S1A_IW_GRDH_1SDV_20240426T154032_20240426T1540...,2024-04-26,-3,True,S1A_IW_GRDH_1SDV_20240226T154031_20240226T1540...,2024-02-26,1.0,1.0,159,ascending,1
1,garissa_pl,2024-05-05,S1A_IW_GRDH_1SDV_20240508T154032_20240508T1540...,2024-05-08,3,True,S1A_IW_GRDH_1SDV_20240226T154031_20240226T1540...,2024-02-26,1.0,1.0,159,ascending,1
2,nairobi_pl,2024-05-01,S1A_IW_GRDH_1SDV_20240501T154818_20240501T1548...,2024-05-01,0,True,S1A_IW_GRDH_1SDV_20240219T154816_20240219T1548...,2024-02-19,1.0,1.0,57,ascending,2


## 3. GRD processing against Earth Engine

Processes the original scenes onto Sen1Floods11 chip grids and compares them with the Earth Engine chips.

In [4]:
import fetch_sen1floods11, validate_grd

fetch_sen1floods11.main()
validate_grd.main()

 ID  location chip_prefix ISO_CC    s1_date    s2_date      orbit  rel_orbit_num  VH_thresh  hand_chips  africa  east_africa
  1   Bolivia     Bolivia    BOL 2018/02/15 2018/02/15 DESCENDING          156.0     -20.44          15   False        False
  2     Ghana       Ghana    GHA 2018/09/18 2018/09/19  ASCENDING          147.0     -22.81          53    True        False
  3     India       India    IND 2016/08/12 2016/08/12 DESCENDING           77.0     -21.56          68   False        False
  4  Cambodia      Mekong    KHM 2018/08/05 2018/08/04  ASCENDING           26.0     -23.06          30   False        False
  5   Nigeria     Nigeria    NGA 2018/09/21 2018/09/20  ASCENDING          103.0     -21.94          18    True        False
  6  Pakistan    Pakistan    PAK 2017/06/28 2017/06/28 DESCENDING            5.0     -19.56          28   False        False
  7  Paraguay    Paraguay    PRY 2018/10/31 2018/10/31 DESCENDING           68.0     -19.94          67   False        False


{'chip': 'Bolivia_379434_S1Hand', 'event': 'Bolivia', 'scene': 'S1B_IW_GRDH_1SDV_20180215T095810_20180215T095835_009632_0115DE', 'pol': 'vv', 'bias_db': 0.0005099773406982422, 'r_zero': 0.9379555998959527, 'r_best': 0.9852559481891711, 'dy': 0, 'dx': -1}
{'chip': 'Bolivia_379434_S1Hand', 'event': 'Bolivia', 'scene': 'S1B_IW_GRDH_1SDV_20180215T095810_20180215T095835_009632_0115DE', 'pol': 'vh', 'bias_db': 0.0008544921875, 'r_zero': 0.9230866240103885, 'r_best': 0.9813250962797162, 'dy': 0, 'dx': -1}


{'chip': 'Bolivia_314919_S1Hand', 'event': 'Bolivia', 'scene': 'S1B_IW_GRDH_1SDV_20180215T095745_20180215T095810_009632_0115DE', 'pol': 'vv', 'bias_db': 0.005013227462768555, 'r_zero': 0.9865232266274121, 'r_best': 0.993768633947548, 'dy': 0, 'dx': -1}
{'chip': 'Bolivia_314919_S1Hand', 'event': 'Bolivia', 'scene': 'S1B_IW_GRDH_1SDV_20180215T095745_20180215T095810_009632_0115DE', 'pol': 'vh', 'bias_db': 0.0021867752075195312, 'r_zero': 0.9825277354316508, 'r_best': 0.9913017031131497, 'dy': 0, 'dx': -1}


{'chip': 'Ghana_154838_S1Hand', 'event': 'Ghana', 'scene': 'S1B_IW_GRDH_1SDV_20180918T181808_20180918T181833_012773_017943', 'pol': 'vv', 'bias_db': -0.0012888908386230469, 'r_zero': 0.9646655244743576, 'r_best': 0.964867474865579, 'dy': 0, 'dx': 0}
{'chip': 'Ghana_154838_S1Hand', 'event': 'Ghana', 'scene': 'S1B_IW_GRDH_1SDV_20180918T181808_20180918T181833_012773_017943', 'pol': 'vh', 'bias_db': 0.0013327598571777344, 'r_zero': 0.9565587997800278, 'r_best': 0.9567436978985133, 'dy': 0, 'dx': 0}


{'chip': 'Ghana_146222_S1Hand', 'event': 'Ghana', 'scene': 'S1B_IW_GRDH_1SDV_20180918T181808_20180918T181833_012773_017943', 'pol': 'vv', 'bias_db': -0.000759124755859375, 'r_zero': 0.9350685458227521, 'r_best': 0.9351441597130734, 'dy': 0, 'dx': 0}
{'chip': 'Ghana_146222_S1Hand', 'event': 'Ghana', 'scene': 'S1B_IW_GRDH_1SDV_20180918T181808_20180918T181833_012773_017943', 'pol': 'vh', 'bias_db': -0.00019216537475585938, 'r_zero': 0.9317224732959942, 'r_best': 0.9320917636182178, 'dy': 0, 'dx': 0}


...

{'chip': 'Spain_8104659_S1Hand', 'event': 'Spain', 'scene': 'S1A_IW_GRDH_1SDV_20190917T060222_20190917T060247_029057_034C21', 'pol': 'vv', 'bias_db': 0.03087615966796875, 'r_zero': 0.9603084392965668, 'r_best': 0.9604231226684498, 'dy': 0, 'dx': 0}
{'chip': 'Spain_8104659_S1Hand', 'event': 'Spain', 'scene': 'S1A_IW_GRDH_1SDV_20190917T060222_20190917T060247_029057_034C21', 'pol': 'vh', 'bias_db': 0.028969764709472656, 'r_zero': 0.9635710207360307, 'r_best': 0.96395956168946, 'dy': 0, 'dx': 0}


{'chip': 'Sri-Lanka_1038087_S1Hand', 'event': 'Sri-Lanka', 'scene': 'S1B_IW_GRDH_1SDV_20170530T002405_20170530T002430_005820_00A340', 'pol': 'vv', 'bias_db': 0.01130533218383789, 'r_zero': 0.9681444474585721, 'r_best': 0.9679539175216934, 'dy': 0, 'dx': 0}
{'chip': 'Sri-Lanka_1038087_S1Hand', 'event': 'Sri-Lanka', 'scene': 'S1B_IW_GRDH_1SDV_20170530T002405_20170530T002430_005820_00A340', 'pol': 'vh', 'bias_db': 0.010416030883789062, 'r_zero': 0.9796293977496094, 'r_best': 0.9796224691284394, 'dy': 0, 'dx': 0}


{'chip': 'Sri-Lanka_748447_S1Hand', 'event': 'Sri-Lanka', 'scene': 'S1B_IW_GRDH_1SDV_20170530T002430_20170530T002501_005820_00A340', 'pol': 'vv', 'bias_db': 0.018342018127441406, 'r_zero': 0.9023796042532201, 'r_best': 0.965029606590737, 'dy': 0, 'dx': -1}
{'chip': 'Sri-Lanka_748447_S1Hand', 'event': 'Sri-Lanka', 'scene': 'S1B_IW_GRDH_1SDV_20170530T002430_20170530T002501_005820_00A340', 'pol': 'vh', 'bias_db': 0.018082618713378906, 'r_zero': 0.9035365582098767, 'r_best': 0.9650995354451191, 'dy': 0, 'dx': -1}


{'chip': 'USA_19225_S1Hand', 'event': 'USA', 'scene': 'S1A_IW_GRDH_1SDV_20190522T002101_20190522T002126_027333_031530', 'pol': 'vv', 'bias_db': 0.011579513549804688, 'r_zero': 0.9798620402836961, 'r_best': 0.9799887260144949, 'dy': 0, 'dx': 0}
{'chip': 'USA_19225_S1Hand', 'event': 'USA', 'scene': 'S1A_IW_GRDH_1SDV_20190522T002101_20190522T002126_027333_031530', 'pol': 'vh', 'bias_db': 0.009243965148925781, 'r_zero': 0.9732807027641457, 'r_best': 0.9732496397484732, 'dy': 0, 'dx': 0}


{'chip': 'USA_788696_S1Hand', 'event': 'USA', 'scene': 'S1A_IW_GRDH_1SDV_20190522T002101_20190522T002126_027333_031530', 'pol': 'vv', 'bias_db': 0.006886482238769531, 'r_zero': 0.995568155682349, 'r_best': 0.9955842667321073, 'dy': 0, 'dx': 0}
{'chip': 'USA_788696_S1Hand', 'event': 'USA', 'scene': 'S1A_IW_GRDH_1SDV_20190522T002101_20190522T002126_027333_031530', 'pol': 'vh', 'bias_db': 0.002178192138671875, 'r_zero': 0.9943800743854994, 'r_best': 0.9943700593924983, 'dy': 0, 'dx': 0}
pol                   vh         vv
bias_db count  18.000000  18.000000
        mean    0.004144   0.006256
        std     0.015084   0.013032
        min    -0.022580  -0.011771
        25%    -0.002630  -0.001156
        50%     0.001094   0.001256
        75%     0.010123   0.011256
        max     0.042895   0.040619
r_zero  count  18.000000  18.000000
        mean    0.955496   0.960399
        std     0.041199   0.039140
        min     0.854388   0.868610
        25%     0.935230   0.943544
       

## 4. Pre-event chips and U-Net training

Pre-event chips for the hand-labelled set, then training. The paper's weights were trained on Kaggle with `kaggle/unet_main` and `kaggle/unet_pre`.

In [5]:
import json, subprocess
import pre_event_chips
from config import MODELS, CACHE

pre_event_chips.main()
if TRAIN_LOCALLY:
    subprocess.run([sys.executable, 'train_unet.py', '--data', str(CACHE / 'sen1floods11'),
                    '--pre-dir', str(pre_event_chips.OUT), '--out', str(MODELS)], cwd=ROOT / 'pipeline', check=True)
results = json.loads((MODELS / 'sen1floods11_results.json').read_text())
pd.DataFrame({run: {f'{split} {kind}': v[split][kind]['mean_chip_iou']
                    for split in ('test', 'bolivia', 'test_paired', 'bolivia_paired') if split in v
                    for kind in ('all_water', 'flood_water')} for run, v in results.items()}).round(3)

{'chip': 'Bolivia_103757', 'event_date': '2018/02/15', 'pre_id': 'S1B_IW_GRDH_1SDV_20180110T095746_20180110T095811_009107_0104AA', 'pre_date': '2018-01-10'}


{'chip': 'Bolivia_129334', 'event_date': '2018/02/15', 'pre_id': None, 'pre_date': None}


{'chip': 'Bolivia_195474', 'event_date': '2018/02/15', 'pre_id': 'S1B_IW_GRDH_1SDV_20180110T095811_20180110T095836_009107_0104AA', 'pre_date': '2018-01-10'}


{'chip': 'Bolivia_23014', 'event_date': '2018/02/15', 'pre_id': None, 'pre_date': None}


{'chip': 'Bolivia_233925', 'event_date': '2018/02/15', 'pre_id': None, 'pre_date': None}


...

{'chip': 'USA_933610', 'event_date': '2019/05/22', 'pre_id': 'S1A_IW_GRDH_1SDV_20190416T002059_20190416T002124_026808_030341', 'pre_date': '2019-04-16'}


{'chip': 'USA_955053', 'event_date': '2019/05/22', 'pre_id': 'S1A_IW_GRDH_1SDV_20190416T002059_20190416T002124_026808_030341', 'pre_date': '2019-04-16'}


{'chip': 'USA_986268', 'event_date': '2019/05/22', 'pre_id': 'S1A_IW_GRDH_1SDV_20190416T002059_20190416T002124_026808_030341', 'pre_date': '2019-04-16'}


{'chip': 'USA_994009', 'event_date': '2019/05/22', 'pre_id': 'S1A_IW_GRDH_1SDV_20190416T002059_20190416T002124_026808_030341', 'pre_date': '2019-04-16'}


,post_hand_weak,post_hand,pre_hand
test all_water,0.424,0.401,NaN
test flood_water,0.358,0.332,NaN
bolivia all_water,0.446,0.329,NaN
bolivia flood_water,0.392,0.306,NaN
test_paired all_water,0.424,0.404,0.409
test_paired flood_water,0.358,0.332,0.337
bolivia_paired all_water,0.446,0.346,0.431
bolivia_paired flood_water,0.392,0.322,0.410


## 5. Kenya input stacks and flood masks

In [6]:
import prepare, methods

for aoi in pd.read_csv(pick_scenes.SCENES)['aoi']:
    prepare.stack(aoi)
methods.main()

          aoi          method  threshold_db
   garissa_s2         otsu_vv    -13.666719
   garissa_s2  change_otsu_vv     -2.139417
   garissa_s2 change_fixed_vv     -3.000000
   garissa_s2         otsu_vh    -20.467318
   garissa_s2  change_otsu_vh     -1.997688
   garissa_s2 change_fixed_vh     -3.000000
   garissa_s2       unet_post           NaN
   garissa_s2  unet_post_hand           NaN
   garissa_s2        unet_pre           NaN
   garissa_pl         otsu_vv    -14.019883
   garissa_pl  change_otsu_vv     -3.504313
   garissa_pl change_fixed_vv     -3.000000
   garissa_pl         otsu_vh    -22.065986
   garissa_pl  change_otsu_vh     -5.955538
   garissa_pl change_fixed_vh     -3.000000
   garissa_pl       unet_post           NaN
   garissa_pl  unet_post_hand           NaN
   garissa_pl        unet_pre           NaN
   nairobi_pl         otsu_vv     -7.642489
   nairobi_pl  change_otsu_vv     -0.651221
   nairobi_pl change_fixed_vv     -3.000000
   nairobi_pl         otsu_vh   

## 6. Agreement with UNOSAT

By AOI, by group, by land cover, with and without the HAND mask, with 1 km block bootstrap intervals.

In [7]:
import evaluate

evaluate.main()

/Users/craigouma/Documents/kenya-floods-sar-2024/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


   group          method  hand_mask   iou  iou_lo  iou_hi    f1  precision  recall
riverine       unet_post      False 0.347   0.331   0.364 0.515      0.530   0.500
riverine        unet_pre      False 0.277   0.262   0.293 0.434      0.392   0.486
riverine  unet_post_hand      False 0.212   0.200   0.225 0.349      0.286   0.448
riverine change_fixed_vv      False 0.160   0.150   0.172 0.276      0.275   0.277
riverine change_fixed_vh      False 0.155   0.144   0.168 0.269      0.212   0.366
riverine         otsu_vh      False 0.116   0.109   0.125 0.209      0.151   0.335
riverine         otsu_vv      False 0.082   0.077   0.088 0.152      0.097   0.351
riverine  change_otsu_vh      False 0.064   0.060   0.069 0.121      0.070   0.430
riverine  change_otsu_vv      False 0.064   0.061   0.069 0.121      0.071   0.414
riverine       unet_post       True 0.347   0.331   0.365 0.515      0.531   0.500
riverine        unet_pre       True 0.279   0.264   0.295 0.436      0.395   0.486
rive

## 7. Day-gap sensitivity and building exposure

In [8]:
import sensitivity, exposure

sensitivity.main()
exposure.main()

using day_gap_sensitivity.csv; pass force=True to recompute


       aoi    group  buildings_in_domain  unosat_structures_in_domain  unosat_structures_layer          method  exposed  ratio_to_unosat
garissa_s2 riverine                84681                         3440                     3440   unosat_extent     2481            0.721
garissa_s2 riverine                84681                         3440                     3440 change_fixed_vh     1919            0.558
garissa_s2 riverine                84681                         3440                     3440 change_fixed_vv     1379            0.401
garissa_s2 riverine                84681                         3440                     3440  change_otsu_vh     5455            1.586
garissa_s2 riverine                84681                         3440                     3440  change_otsu_vv     3734            1.085
garissa_s2 riverine                84681                         3440                     3440         otsu_vh      797            0.232
garissa_s2 riverine                84681 

## 8. Lower Tana flood time series

In [9]:
import tana_timeseries

TANA_METHOD = (ROOT / 'outputs' / 'tana_method.txt').read_text().strip()
tana_timeseries.build(TANA_METHOD)

I1010 14:46:41.694796 10583428 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I1010 14:46:41.723147 10613248 ev_poll_posix.cc:587] FD from fork parent still in poll list: fd(103, generation: 1)


I1010 14:46:42.052406 10583428 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


I1010 14:46:42.076823 10583428 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I1010 14:46:42.082582 10613263 ev_poll_posix.cc:587] FD from fork parent still in poll list: fd(103, generation: 1)
I1010 14:46:42.107125 10613278 ev_poll_posix.cc:587] FD from fork parent still in poll list: fd(103, generation: 1)


garissa_s2 7 passes
tana_500_9660 7 passes


tana_500_9920 7 passes
tana_500_9900 7 passes


...

tana_660_9820 7 passes


tana_680_9720 7 passes


tana_680_9820 7 passes


tana_680_9840 7 passes


,orbit,date,county,observed_km2,flooded_km2
0,57,2024-04-07,Garissa,0.0000,0.0000
1,57,2024-04-07,Tana River,204.5960,19.3310
2,57,2024-04-19,Garissa,0.0000,0.0000
3,57,2024-04-19,Tana River,204.5960,29.5803
4,57,2024-05-01,Garissa,0.0000,0.0000
5,57,2024-05-01,Tana River,204.5960,62.6490
6,57,2024-05-25,Garissa,0.0000,0.0000
7,57,2024-05-25,Tana River,204.5960,95.0952
8,57,2024-06-06,Garissa,0.0000,0.0000
9,57,2024-06-06,Tana River,204.5960,111.7737


## 9. Figures

In [10]:
import figures

figures.main()